# 5.2 Bagging - Bootstrap AGGregratING 自举汇聚法

- 独立训练 n 个模型，每个模型都是在 bootstrap 采样的数据上训练的
- 使用 n 个模型的结果取均值(回归)或投票(分类)
- **降低模型的方差**

### bootstrap sample
- 可重复随机采样
- 每次 bootstrap sample 可以采样到约 $1 - \frac{1}{e} \approx 63\%$ unique examples，剩下没有被采样到的样本可以作为验证集

In [ ]:
class Bagging:
    def __init__(self, base_learner, n_learners):
        self.learners = [clone(base_learner) for _ in range(n_learners)]

    def fit(self, X, y):
        for learner in self.learners:
            examples = np.random.choice(
                np.arange(len(X)), int(len(X)), replace=True)
            learner.fit(X.iloc[examples, :], y.iloc[examples])

    def predict(self, X):
        preds = [learner.predict(X) for learner in self.learners]
        return np.array(preds).mean(axis=0)

### Random Forest
- 使用决策树作为 base learner
- 每次 bootstrap 采样的是特征维度（列），而非样本，且不开启 replacement（重复特征没意义）

通常不会增大泛化误差：减小了模型方差，且没有增加偏差

<img src="paste_image/2026-01-29-16-27-10.png" width="68%">

### Unstable Learners
应当使用不稳定的模型作为 base learner
- Consider regression for simplicity, given ground truth $f$ and base learner $h$, bagging: $\hat{f}(x) = \mathbb{E}[h(x)]$

- Given $\left( \mathbb{E}[x] \right)^2 \leq \mathbb{E}[x^2]$, we have 

  $$\left( f(x) - \hat{f}(x) \right)^2 \leq \mathbb{E}\left[ \left( f(x) - h(x) \right)^2 \right] \iff \left( \mathbb{E}[h(x)] \right)^2 \leq \mathbb{E}[h(x)^2]$$

由柯西不等式：取等条件为 所有的h都相等

h 之间差别越大，则变小的效果越好